In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from natsort import natsorted
import matplotlib as mpl

# Add function path
original_path = os.getcwd()
os.chdir(original_path)
function_path = './functions/'
sys.path.append(function_path)

# Import custom functions
from analysis_function import *
from kcc_constrain_function import *
from Plot_function import *
from f1_0_H_sum_multi_region_constrain import *
from f_IMP import *
from f_schemes import *
from ar6_area_weighted_cont_global import *


In [41]:
path = './saved_data/'

def load_xr_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)

obs_all = load_xr_pickle(path + '0.1.HadCRUT5.Tas.anomalies.46AR6regions_7cont_1glob_175years_1850-2024.pkl')

obs_200runs_all = load_xr_pickle(path + '0.2.HadCRUT5.200runs.nonmasked.Tas.anomalies.46AR6regions_7cont_1glob_175years_1850-2024.pkl')

mod_all_his_fu = load_xr_pickle(path + '0.4.Smoothed_His-Fu-ALL.25mods.mean.nonmasked.Tas.anomalies.46AR6regions_7cont_1glob_251years_1850-2100.pkl')

ln_mod_all = load_xr_pickle(path + '0.3.Large_ensembles.320runs.nonmasked.Tas.anomalies.46AR6regions_7cont_1glob_251years_1850-2100.pkl')

mod_45_pseudo = load_xr_pickle(path + '0.5.pseudo-model.15mod.run1-3.nonmasked.Tas.anomalies.46AR6regions_7cont_1glob_176years_1850-2025.pkl')

print_xarray_info(obs_all, obs_200runs_all, mod_all_his_fu, ln_mod_all, mod_45_pseudo)


Array 1:
  Sizes: Frozen({'year': 175, 'region': 54})
  Coords: ['realization', 'year', 'region', 'abbrevs', 'names']

Array 2:
  Sizes: Frozen({'runs': 200, 'year': 175, 'region': 54})
  Coords: ['runs', 'year', 'region', 'abbrevs', 'names']

Array 3:
  Sizes: Frozen({'forcing': 1, 'model_name': 25, 'year': 251, 'region': 54})
  Coords: ['model_name', 'region', 'year', 'abbrevs', 'names', 'forcing']

Array 4:
  Sizes: Frozen({'model_run': 320, 'year': 251, 'region': 54})
  Coords: ['year', 'model_run', 'model_name', 'region', 'abbrevs', 'names']

Array 5:
  Sizes: Frozen({'model_run': 45, 'year': 176, 'region': 54})
  Coords: ['model_run', 'year', 'model_name', 'region', 'abbrevs', 'names']


In [51]:
from f_schemes import *

# -----------------------------------------
# Define time periods
# -----------------------------------------

ref_period = (1850, 1900)

warming_target_period1 = (2016, 2025)

# -----------------------------------------
# Define constraint schemes
# -----------------------------------------
region_groups = {
    'North and Central America': ['GIC', 'NWN', 'NEN', 'WNA', 'CNA', 'ENA', 'NCA', 'SCA', 'CAR'],
    'South America': ['NWS', 'NSA', 'NES', 'SAM', 'SWS', 'SES', 'SSA'],
    'Europe': ['NEU', 'WCE', 'EEU', 'MED'],
    'Africa': ['SAH', 'WAF', 'CAF', 'NEAF', 'SEAF', 'WSAF', 'ESAF', 'MDG'],
    'Asia': ['RAR', 'WSB', 'ESB', 'RFE', 'WCA', 'ECA', 'TIB', 'EAS', 'ARP', 'SAS'],
    'Australasia': ['SEA', 'NAU', 'CAU', 'EAU', 'SAU', 'NZ'],
    'Antarctica': ['EAN', 'WAN'],
    # 'Global land': ['LSAT']
}



In [52]:
scheme_dict = {
    "single_region": generate_single_region_constraint_pairs(region_groups),
    "region_within_continent": generate_continent_allregion_constraint_pairs(region_groups),
    "continent_mean": generate_continent_mean_constraint_pairs(region_groups, continent_abbrevs),
    "continent_region": generate_continent_regional_target_constraint_pairs(region_groups, continent_abbrevs)
}

In [ ]:
scheme_dict

### Constrained warming relative to current year 2025

In [53]:
mod_all_his_fu_his= mod_all_his_fu
#### global_region constraint
mod_all_his_fu_da = mod_all_his_fu.isel(region=slice(0, 46))

constrain_func = constrain_sum_reg
obs = obs_all
obs_200runs = obs_200runs_all
ln_mod = ln_mod_all
mod_his = mod_all_his_fu_his
mod_da = mod_all_his_fu_da
obs_ar6 = obs_all
region_names = list(mod_da.names.values)
forcing_list = mod_da.forcing  # xarray.DataArray
his_forcing = ['ALL']
constrain_forcing_names=['ALL']


post_list = []
scheme_names = []

for scheme_name, scheme_pairs in scheme_dict.items():
    
    _, _, _, post_smooth_ALL_ref2025, _ = \
        process_all_regions(scheme_pairs, constrain_func, obs, obs_200runs, ln_mod, mod_his, mod_da, obs_ar6, region_names, forcing_list, his_forcing, constrain_forcing_names, 
        reg_id = slice(0, len(region_names)),
        uncertainty_ref_period=(1850, 2025),
        ref_period=(2025, 2026),
        obs_adjust_ref_period=(1961, 2025),
        warming_target_period=(2016, 2025),
        calc_smoothed = True, 
        print_constraint_regions = True)

    # add scheme dim
    post_smooth_ALL_ref2025 = post_smooth_ALL_ref2025.expand_dims(scheme=[scheme_name])

    post_list.append(post_smooth_ALL_ref2025)
    scheme_names.append(scheme_name)

# combine all schemes
post_smooth_ALL_ref2025_sche = xr.concat(post_list, dim="scheme")

Processing region 0: Greenland/Iceland
 Constraining used region 0: GIC
Processing region 1: N.W.North-America
 Constraining used region 1: NWN
Processing region 2: N.E.North-America
 Constraining used region 2: NEN
Processing region 3: W.North-America
 Constraining used region 3: WNA
Processing region 4: C.North-America
 Constraining used region 4: CNA
Processing region 5: E.North-America
 Constraining used region 5: ENA
Processing region 6: N.Central-America
 Constraining used region 6: NCA
Processing region 7: S.Central-America
 Constraining used region 7: SCA
Processing region 8: Caribbean
 Constraining used region 8: CAR
Processing region 9: N.W.South-America
 Constraining used region 9: NWS
Processing region 10: N.South-America
 Constraining used region 10: NSA
Processing region 11: N.E.South-America
 Constraining used region 11: NES
Processing region 12: South-American-Monsoon
 Constraining used region 12: SAM
Processing region 13: S.W.South-America
 Constraining used region 13:

In [54]:
post_smooth_ALL_ref2025_sche

<xarray.DataArray 'post_mean_5_95_smooth' (scheme: 4, region: 46, forcing: 1,
                                           year: 251, quantile: 3)> Size: 1MB
array([[[[[-2.64498549, -3.32175374, -1.9731544 ],
          [-2.64735656, -3.3162119 , -1.98291186],
          [-2.64972795, -3.31067108, -1.99266367],
          ...,
          [ 5.04833975,  1.01485899,  9.00134997],
          [ 5.11797202,  1.04858257,  9.10457711],
          [ 5.18760429,  1.08230615,  9.20780426]]],


        [[[-2.94684156, -3.41233645, -2.4986663 ],
          [-2.94705554, -3.40429901, -2.50403898],
          [-2.94727218, -3.39627534, -2.50941341],
          ...,
          [ 5.90049119,  2.81333035,  8.86735485],
          [ 5.98187727,  2.86386925,  8.97598569],
          [ 6.06326336,  2.91440814,  9.08461653]]],


        [[[-3.19970696, -3.84302353, -2.54103001],
          [-3.19606488, -3.83132341, -2.54752723],
...
          [ 2.83633797,  1.99605796,  3.64111954]]],


        [[[-1.69804492, -1.95662209, -1.43940961],
          [-1.69491675, -1.95652953, -1.43300258],
          [-1.69179008, -1.95643696, -1.42659477],
          ...,
          [ 3.56522832,  1.16680203,  5.87909605],
          [ 3.61440388,  1.19165206,  5.95005887],
          [ 3.66357945,  1.21650209,  6.02102169]]],


        [[[-1.86182645, -2.43580871, -1.2783018 ],
          [-1.85727101, -2.42147943, -1.28469228],
          [-1.8527156 , -2.40715482, -1.29107684],
          ...,
          [ 3.57239613,  0.27548481,  6.88942752],
          [ 3.62167056,  0.29154055,  6.97267387],
          [ 3.67094499,  0.3075963 ,  7.05592022]]]]],
      shape=(4, 46, 1, 251, 3))
Coordinates:
  * scheme       (scheme) object 32B 'single_region' ... 'continent_region'
  * region       (region) int64 368B 0 1 2 3 4 5 6 7 ... 38 39 40 41 42 43 44 45
  * year         (year) int64 2kB 1850 1851 1852 1853 ... 2097 2098 2099 2100
  * forcing      (forcing) <U3 12B 'ALL'
  * quantile     (quantile) <U4 48B 'mean' '5th' '95th'
    abbrevs      (region) <U4 736B 'GIC' 'NWN' 'NEN' 'WNA' ... 'NZ' 'EAN' 'WAN'
    names        (region) <U25 5kB 'Greenland/Iceland' ... 'W.Antarctica'
    realization  int64 8B 100

### Constrained warming relative to pre-industrial 1850-1900 year

In [55]:
mod_all_his_fu_his= mod_all_his_fu
#### global_region constraint
mod_all_his_fu_da = mod_all_his_fu.isel(region=slice(0, 46))

constrain_func = constrain_sum_reg
obs = obs_all
obs_200runs = obs_200runs_all
ln_mod = ln_mod_all
mod_his = mod_all_his_fu_his
mod_da = mod_all_his_fu_da
obs_ar6 = obs_all
region_names = list(mod_da.names.values)
forcing_list = mod_da.forcing  # xarray.DataArray
his_forcing = ['ALL']
constrain_forcing_names=['ALL']


post_list = []
scheme_names = []

for scheme_name, scheme_pairs in scheme_dict.items():
    
    _, _, _, post_smooth_ALL_ref1850, _ = \
        process_all_regions(scheme_pairs, constrain_func, obs, obs_200runs, ln_mod, mod_his, mod_da, obs_ar6, region_names, forcing_list, his_forcing, constrain_forcing_names, 
        reg_id = slice(0, len(region_names)),
        uncertainty_ref_period=(1850, 2025),
        ref_period=(1850, 1900),
        obs_adjust_ref_period=(1961, 2025),
        warming_target_period=(2016, 2025),
        calc_smoothed = True, 
        print_constraint_regions = True)

    # add scheme dim
    post_smooth_ALL_ref1850 = post_smooth_ALL_ref1850.expand_dims(scheme=[scheme_name])

    post_list.append(post_smooth_ALL_ref1850)
    scheme_names.append(scheme_name)

# combine all schemes
post_smooth_ALL_ref1850_sche = xr.concat(post_list, dim="scheme")

Processing region 0: Greenland/Iceland
 Constraining used region 0: GIC
Processing region 1: N.W.North-America
 Constraining used region 1: NWN
Processing region 2: N.E.North-America
 Constraining used region 2: NEN
Processing region 3: W.North-America
 Constraining used region 3: WNA
Processing region 4: C.North-America
 Constraining used region 4: CNA
Processing region 5: E.North-America
 Constraining used region 5: ENA
Processing region 6: N.Central-America
 Constraining used region 6: NCA
Processing region 7: S.Central-America
 Constraining used region 7: SCA
Processing region 8: Caribbean
 Constraining used region 8: CAR
Processing region 9: N.W.South-America
 Constraining used region 9: NWS
Processing region 10: N.South-America
 Constraining used region 10: NSA
Processing region 11: N.E.South-America
 Constraining used region 11: NES
Processing region 12: South-American-Monsoon
 Constraining used region 12: SAM
Processing region 13: S.W.South-America
 Constraining used region 13:

In [57]:
save_path = './saved_data/'
name = '4.0_constrained_46reg_4schemes_smoothed_warming_ref2025_1850-2100.pkl'

with open(save_path + name, 'wb') as wi:
	pickle.dump(post_smooth_ALL_ref2025_sche, wi)

name = '4.0_constrained_46reg_4schemes_smoothed_warming_ref1850_1850-2100.pkl'

with open(save_path + name, 'wb') as wi:
	pickle.dump(post_smooth_ALL_ref1850_sche, wi)
